# S10 - Unidad 1 - Evidencia de Aprendizaje 1 - Carga y Creación de Dataset

Este Notebook actúa como **orquestador** de la actividad:

```
Hugging Face → HFDatasetLoader → DataFrame → 4 transformaciones → df_final → DatasetSummarizer → resumen_dataset.txt
```

La lógica vive en las clases del paquete `src/`; aquí solo se coordinan.

## 1. Importación de las clases

Se agrega la carpeta raíz de la actividad al `sys.path` para poder importar el paquete `src`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.hf_dataset_loader import HFDatasetLoader
from src.dataset_summarizer import DatasetSummarizer

print("Raíz del proyecto:", PROJECT_ROOT)
print("Clases importadas:", HFDatasetLoader.__name__, "y", DatasetSummarizer.__name__)

Raíz del proyecto: C:\Proyectos\MachineLearningIU\S10_Unidad1_Evidencia1_Carga_y_Creacion_de_Dataset
Clases importadas: HFDatasetLoader y DatasetSummarizer


## 2. Carga del dataset desde Hugging Face

Se crea el `HFDatasetLoader` con el identificador `scikit-learn/iris` y el split `train`
(el único que ofrece este dataset). El método `load()` descarga los datos con la librería
`datasets` y los devuelve como `pandas.DataFrame`.

In [2]:
DATASET_ID = "scikit-learn/iris"

loader = HFDatasetLoader(DATASET_ID, split="train")
df = loader.load()

print("Dataset cargado:", DATASET_ID)
print("Tipo de objeto:", type(df))
print("Dimensiones (filas, columnas):", df.shape)

Dataset cargado: scikit-learn/iris
Tipo de objeto: <class 'pandas.DataFrame'>
Dimensiones (filas, columnas): (150, 6)


## 3. Información inicial del DataFrame original

In [3]:
df.head(10)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
5,6,5.4,3.9,1.7,0.4,Iris-setosa
6,7,4.6,3.4,1.4,0.3,Iris-setosa
7,8,5.0,3.4,1.5,0.2,Iris-setosa
8,9,4.4,2.9,1.4,0.2,Iris-setosa
9,10,4.9,3.1,1.5,0.1,Iris-setosa


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    str    
dtypes: float64(4), int64(1), str(1)
memory usage: 9.1 KB


In [5]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

## 4. Transformaciones

### Transformación 1 — Renombrar columnas

Las columnas originales usan *CamelCase* con la unidad incluida (`SepalLengthCm`, `PetalWidthCm`, ...).
Se renombran a **snake_case** en minúsculas (`sepal_length`, `petal_width`, ...) para que sean
claras, consistentes y cómodas de usar en código. Todas las medidas están en centímetros.

In [6]:
column_mapping = {
    "Id": "id",
    "SepalLengthCm": "sepal_length",
    "SepalWidthCm": "sepal_width",
    "PetalLengthCm": "petal_length",
    "PetalWidthCm": "petal_width",
    "Species": "species",
}

df_renamed = df.rename(columns=column_mapping)

print("Antes:  ", list(df.columns))
print("Después:", list(df_renamed.columns))
df_renamed.head()

Antes:   ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']
Después: ['id', 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


,id,sepal_length,sepal_width,petal_length,petal_width,species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Transformación 2 — Seleccionar columnas relevantes

Se conservan solo las cuatro medidas de la flor y la especie. La columna `id` se descarta
porque es un número consecutivo que identifica la fila pero **no aporta información** sobre
la flor (sus estadísticas, como la media, no tendrían significado en el resumen).

In [7]:
selected_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

df_selected = df_renamed[selected_columns].copy()

print("Columnas seleccionadas:", list(df_selected.columns))
print("Dimensiones:", df_selected.shape)
df_selected.head()

Columnas seleccionadas: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
Dimensiones: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Transformación 3 — Filtrar registros

Se conservan únicamente las flores cuyo **largo de sépalo es de al menos 5.0 cm**
(`sepal_length >= 5.0`). Es una condición numérica sencilla que deja fuera los ejemplares
más pequeños. Se muestran las filas antes y después, y cuántas flores quedan por especie.

In [8]:
MIN_SEPAL_LENGTH = 5.0

df_filtered = df_selected[df_selected["sepal_length"] >= MIN_SEPAL_LENGTH].reset_index(drop=True)

print(f"Filas antes del filtro:   {len(df_selected)}")
print(f"Filas después del filtro: {len(df_filtered)}")
print("Largo de sépalo mínimo resultante:", df_filtered["sepal_length"].min())
print()
print(df_filtered["species"].value_counts())
df_filtered.head()

Filas antes del filtro:   150
Filas después del filtro: 128
Largo de sépalo mínimo resultante: 5.0

species
Iris-versicolor    49
Iris-virginica     49
Iris-setosa        30
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,5.0,3.6,1.4,0.2,Iris-setosa
2,5.4,3.9,1.7,0.4,Iris-setosa
3,5.0,3.4,1.5,0.2,Iris-setosa
4,5.4,3.7,1.5,0.2,Iris-setosa


### Transformación 4 — Crear una columna derivada

Se crea `petal_area = petal_length * petal_width`. Representa una **aproximación del área
del pétalo** en cm² (como si fuera un rectángulo). Resume en un solo número el tamaño del
pétalo y es útil para comparar especies.

In [9]:
df_final = df_filtered.copy()
df_final["petal_area"] = (df_final["petal_length"] * df_final["petal_width"]).round(2)

df_final.head()

,sepal_length,sepal_width,petal_length,petal_width,species,petal_area
0,5.1,3.5,1.4,0.2,Iris-setosa,0.28
1,5.0,3.6,1.4,0.2,Iris-setosa,0.28
2,5.4,3.9,1.7,0.4,Iris-setosa,0.68
3,5.0,3.4,1.5,0.2,Iris-setosa,0.30
4,5.4,3.7,1.5,0.2,Iris-setosa,0.30


In [10]:
# Área media del pétalo por especie: muestra la utilidad de la nueva variable
df_final.groupby("species")["petal_area"].mean().round(2)

species
Iris-setosa         0.42
Iris-versicolor     5.77
Iris-virginica     11.37
Name: petal_area, dtype: float64

## 5. DataFrame final

`df_final` es el resultado de aplicar las cuatro transformaciones y es el que se envía a
`DatasetSummarizer`.

In [11]:
print("Dimensiones de df_final:", df_final.shape)
print("Columnas:", list(df_final.columns))
df_final.info()

Dimensiones de df_final: (128, 6)
Columnas: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species', 'petal_area']
<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  128 non-null    float64
 1   sepal_width   128 non-null    float64
 2   petal_length  128 non-null    float64
 3   petal_width   128 non-null    float64
 4   species       128 non-null    str    
 5   petal_area    128 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.9 KB


In [12]:
df_final.describe()

,sepal_length,sepal_width,petal_length,petal_width,petal_area
count,128.000000,128.000000,128.000000,128.000000,128.000000
mean,6.041406,3.053906,4.121875,1.353906,6.661172
std,0.727040,0.451751,1.626887,0.704195,4.519295
min,5.000000,2.000000,1.200000,0.100000,0.150000
25%,5.500000,2.800000,3.450000,1.000000,3.450000
50%,6.000000,3.000000,4.500000,1.400000,6.650000
75%,6.500000,3.325000,5.225000,1.900000,10.072500
max,7.900000,4.400000,6.900000,2.500000,15.870000


## 6. Generación del resumen con `DatasetSummarizer`

Se crea el `DatasetSummarizer` con `df_final` y se guarda el resumen en
`output/resumen_dataset.txt`.

In [13]:
OUTPUT_PATH = PROJECT_ROOT / "output" / "resumen_dataset.txt"

summarizer = DatasetSummarizer(df_final, title="Resumen del dataset Iris transformado")
saved_path = summarizer.save(OUTPUT_PATH)

print("Resumen guardado en:", saved_path)

Resumen guardado en: C:\Proyectos\MachineLearningIU\S10_Unidad1_Evidencia1_Carga_y_Creacion_de_Dataset\output\resumen_dataset.txt


## 7. Comprobación del archivo generado

In [14]:
print("¿Existe el archivo?", saved_path.exists())
print("Tamaño (bytes):", saved_path.stat().st_size)

summary_text = saved_path.read_text(encoding="utf-8")

# El resumen debe corresponder al DataFrame FINAL, no al original
assert f"Cantidad de filas:    {len(df_final)}" in summary_text
assert f"Cantidad de columnas: {df_final.shape[1]}" in summary_text
assert "petal_area" in summary_text
print("Verificado: el resumen corresponde a df_final",
      f"({len(df_final)} filas y {df_final.shape[1]} columnas; original: {df.shape}).")

¿Existe el archivo? True
Tamaño (bytes): 1819
Verificado: el resumen corresponde a df_final (128 filas y 6 columnas; original: (150, 6)).


## 8. Contenido del resumen

In [15]:
print(summary_text)

RESUMEN DEL DATASET IRIS TRANSFORMADO

Cantidad de filas:    128
Cantidad de columnas: 6

COLUMNAS Y TIPOS DE DATO
----------------------------------------------------------------------
  sepal_length         float64
  sepal_width          float64
  petal_length         float64
  petal_width          float64
  species              str
  petal_area           float64

VALORES NULOS POR COLUMNA
----------------------------------------------------------------------
  sepal_length         0
  sepal_width          0
  petal_length         0
  petal_width          0
  species              0
  petal_area           0

ESTADÍSTICAS DESCRIPTIVAS (columnas numéricas)
----------------------------------------------------------------------
       sepal_length  sepal_width  petal_length  petal_width  petal_area
count       128.000      128.000       128.000      128.000     128.000
mean          6.041        3.054         4.122        1.354       6.661
std           0.727        0.452         1.627   

## 9. Conclusiones

| # | Transformación | Operación de pandas | Efecto |
|---|----------------|--------------------|--------|
| 1 | Renombrar columnas | `rename` | Nombres claros en snake_case |
| 2 | Seleccionar columnas | selección por lista | Se descarta `id` |
| 3 | Filtrar registros | máscara booleana | Solo flores con `sepal_length >= 5.0` |
| 4 | Columna derivada | operación aritmética | Nueva variable `petal_area` |

El flujo completo funcionó: los datos se obtuvieron desde Hugging Face con `HFDatasetLoader`,
se transformaron en el Notebook y `DatasetSummarizer` generó el resumen del DataFrame final.